# Practice Lab Assignment 1 — Neural Network Implementation from Scratch
### T.Y. Tech — Generative AI Lab
### Department of CSE (AIML), MIT Academy of Engineering, Alandi, Pune

**Name:** _[Your Full Name]_
**PRN:** _[Your PRN Number]_
**Batch:** _[Your Batch]_
**Date of Submission:** _[Date]_


## 1. Objective

To implement a simple feedforward neural network **from scratch** in Python, without using any in-built
deep learning libraries (no TensorFlow, PyTorch `nn` module, or Keras). The implementation covers three
core components:

- **Forward pass** — computing weighted sums and applying activation functions layer by layer
- **Backpropagation** — deriving and computing gradients of the loss w.r.t. every weight and bias using the chain rule
- **Training via gradient descent** — iteratively updating parameters to minimize the loss


## 2. Problem Definition

**Dataset:** EMNIST — Letters split (Extended MNIST). It contains 28x28 grayscale images of handwritten
English letters (A–Z), in the same format as the classic MNIST digit dataset, but with 26 balanced classes.

**Task:** Multi-class image classification — given a flattened 784-pixel grayscale image, predict which
letter of the alphabet (A–Z) it represents.


## 3. Methodology

### 3.1 Neural Network Architecture

| Layer  | Size                          | Activation |
|--------|-------------------------------|------------|
| Input  | 784 neurons (28x28 flattened) | —          |
| Hidden | 128 neurons                   | ReLU       |
| Output | 26 neurons (A–Z)              | Softmax    |

Weights are initialized with small random values (scaled by 0.01) to break symmetry between neurons
while keeping initial activations small. Biases are initialized to zero.

### 3.2 Forward Pass

```
Z1 = X . W1 + b1        (hidden pre-activation)
A1 = ReLU(Z1)            (hidden activation)
Z2 = A1 . W2 + b2        (output pre-activation)
A2 = Softmax(Z2)          (predicted class probabilities)
```

### 3.3 Loss Function

Categorical Cross-Entropy, the standard loss for multi-class classification with a softmax output layer:

```
L = -(1/N) * sum( Y_true * log(A2) )
```

### 3.4 Backpropagation

Gradients are derived via the chain rule and propagated backward from the output layer to the input layer:

```
dZ2 = A2 - Y_true                    (softmax + cross-entropy combined gradient)
dW2 = A1^T . dZ2 / N,  db2 = mean(dZ2)
dA1 = dZ2 . W2^T
dZ1 = dA1 * ReLU'(Z1)
dW1 = X^T . dZ1 / N,  db1 = mean(dZ1)
```

### 3.5 Optimization

Full-batch Gradient Descent — the entire training subsample is passed through the network every epoch.
Each parameter is updated as:

```
W = W - learning_rate * dW
b = b - learning_rate * db
```


## 4. Implementation

### 4.1 Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

np.random.seed(42)  # for reproducibility


### 4.2 Data Loading

EMNIST Letters is loaded via `torchvision.datasets.EMNIST`, which downloads the dataset directly from
NIST's servers. Note: this cell requires internet access and may take a minute the first time it runs.

In [ ]:
train_raw = datasets.EMNIST(root='./data', split='letters', train=True, download=True)
test_raw = datasets.EMNIST(root='./data', split='letters', train=False, download=True)

print("Raw train set size:", len(train_raw))
print("Raw test set size:", len(test_raw))


### 4.3 Preprocessing

- Images are flattened from 28x28 to a 784-length vector
- Pixel values are normalized to the [0, 1] range
- Labels are shifted from EMNIST's native 1–26 range to 0–25, then one-hot encoded
- A subsample is used so training is fast on plain NumPy/CPU


In [ ]:
def to_numpy_dataset(torch_dataset, n_samples):
    n_samples = min(n_samples, len(torch_dataset))
    X = np.zeros((n_samples, 784), dtype=np.float32)
    y = np.zeros((n_samples,), dtype=np.int64)
    for i in range(n_samples):
        img, label = torch_dataset[i]
        # EMNIST images are stored transposed relative to visual orientation
        arr = np.array(img, dtype=np.float32).T
        X[i] = arr.flatten() / 255.0
        y[i] = label - 1  # shift 1-26 -> 0-25
    return X, y

N_TRAIN = 8000
N_TEST = 1500

X_train, y_train_labels = to_numpy_dataset(train_raw, N_TRAIN)
X_test, y_test_labels = to_numpy_dataset(test_raw, N_TEST)

def one_hot(labels, n_classes=26):
    oh = np.zeros((labels.shape[0], n_classes))
    oh[np.arange(labels.shape[0]), labels] = 1
    return oh

Y_train = one_hot(y_train_labels)
Y_test = one_hot(y_test_labels)

print("X_train:", X_train.shape, " Y_train:", Y_train.shape)
print("X_test:", X_test.shape, " Y_test:", Y_test.shape)


### 4.4 Activation Functions

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def softmax(Z):
    # subtract row-wise max for numerical stability
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)


### 4.5 Loss Function

In [ ]:
def cross_entropy_loss(Y_true, A2):
    N = Y_true.shape[0]
    A2_clipped = np.clip(A2, 1e-12, 1 - 1e-12)  # avoid log(0)
    loss = -np.sum(Y_true * np.log(A2_clipped)) / N
    return loss


### 4.6 Weight Initialization

In [ ]:
INPUT_SIZE = 784
HIDDEN_SIZE = 128
OUTPUT_SIZE = 26

def initialize_parameters():
    W1 = np.random.randn(INPUT_SIZE, HIDDEN_SIZE) * 0.01
    b1 = np.zeros((1, HIDDEN_SIZE))
    W2 = np.random.randn(HIDDEN_SIZE, OUTPUT_SIZE) * 0.01
    b2 = np.zeros((1, OUTPUT_SIZE))
    return W1, b1, W2, b2


### 4.7 Forward Pass

In [ ]:
def forward_pass(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)
    cache = (Z1, A1, Z2, A2)
    return A2, cache


### 4.8 Backward Pass

In [ ]:
def backward_pass(X, Y_true, W2, cache):
    Z1, A1, Z2, A2 = cache
    N = X.shape[0]

    dZ2 = A2 - Y_true
    dW2 = A1.T @ dZ2 / N
    db2 = np.sum(dZ2, axis=0, keepdims=True) / N

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = X.T @ dZ1 / N
    db1 = np.sum(dZ1, axis=0, keepdims=True) / N

    return dW1, db1, dW2, db2


### 4.9 Training Loop

In [ ]:
def train(X, Y, epochs=200, learning_rate=0.3):
    W1, b1, W2, b2 = initialize_parameters()
    losses = []

    for epoch in range(epochs):
        A2, cache = forward_pass(X, W1, b1, W2, b2)
        loss = cross_entropy_loss(Y, A2)
        losses.append(loss)

        dW1, db1, dW2, db2 = backward_pass(X, Y, W2, cache)

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        if epoch % 20 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

    return W1, b1, W2, b2, losses

EPOCHS = 200
LEARNING_RATE = 0.3

W1, b1, W2, b2, losses = train(X_train, Y_train, epochs=EPOCHS, learning_rate=LEARNING_RATE)


### 4.10 Training Loss Curve

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()


### 4.11 Evaluation

In [ ]:
def predict(X, W1, b1, W2, b2):
    A2, _ = forward_pass(X, W1, b1, W2, b2)
    return np.argmax(A2, axis=1)

def accuracy(X, y_labels, W1, b1, W2, b2):
    preds = predict(X, W1, b1, W2, b2)
    return np.mean(preds == y_labels)

train_acc = accuracy(X_train, y_train_labels, W1, b1, W2, b2)
test_acc = accuracy(X_test, y_test_labels, W1, b1, W2, b2)

print(f"Final Train Accuracy: {train_acc*100:.2f}%")
print(f"Final Test Accuracy:  {test_acc*100:.2f}%")


### 4.12 Sample Predictions

In [ ]:
letters = [chr(ord('A') + i) for i in range(26)]

n_show = 5
idx = np.random.choice(len(X_test), n_show, replace=False)
preds = predict(X_test[idx], W1, b1, W2, b2)

fig, axes = plt.subplots(1, n_show, figsize=(12, 3))
for i, ax in enumerate(axes):
    img = X_test[idx[i]].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"True: {letters[y_test_labels[idx[i]]]}\nPred: {letters[preds[i]]}")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Results

_After running all cells above, summarize your actual results here, for example:_

- **Final Train Accuracy:** _[fill in from your run]_
- **Final Test Accuracy:** _[fill in from your run]_
- Briefly comment on the training loss curve (did it decrease smoothly? plateau?) and on any
  misclassified sample predictions and why the model might have confused those letters.


## 6. Conclusion

_[Write 3-5 sentences summarizing what was implemented, the architecture used (784 → 128 ReLU → 26 Softmax),
and your final train/test accuracy. Mention what you observed about training stability and any
misclassifications.]_

**Possible extensions:** training on the full EMNIST Letters set (~124,800 images), adding a second
hidden layer, using an adaptive optimizer such as Adam, switching to mini-batch gradient descent, or
applying data augmentation.


## Declaration

I, **[Your Full Name]**, confirm that the work submitted in this assignment is my own and has been
completed following academic integrity guidelines. The code is uploaded on my GitHub repository account,
and the repository link is provided below:

**GitHub Repository Link:** [Insert your GitHub link here]

**Signature:** [Your Full Name]


## Submission Checklist

- [ ] Code file (this Jupyter Notebook)
- [ ] Dataset or link to the dataset (EMNIST Letters, via `torchvision.datasets.EMNIST`)
- [ ] Visualizations (loss curve, sample predictions) — included above
- [ ] Screenshots of model performance metrics
- [ ] Readme file
